# SIGMA v2 — training ulang tanpa menambah foto
Perbedaan dari v1 (lihat `potong_lot.py`):
- **Potongan resolusi asli 1664 px** di sekitar benih, dilatih pada imgsz 1664 → benih ±110 px (v1 ±47 px), skala sama di semua foto.
- **copy-paste** (benih ditempel ke foto lain) menambah contoh cacat, `cls` dinaikkan, patience lebih panjang.
- Dua arsitektur dilatih (YOLO26s-seg, YOLO11s-seg); dipilih yang terbaik di **valid**, test hanya untuk laporan.
- Penghitungan per lot memakai pencocokan per benih dan membuang deteksi ganda.

Dataset diambil dari `MyDrive/sigma-v2-potong.zip` (keluaran `potong_lot.py`).

In [ ]:
!nvidia-smi -L
%pip install -q -U ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, pathlib, yaml
SIMPAN = '/content/drive/MyDrive/sigma-benih/runs'
ZIP = next(p for p in ['/content/drive/MyDrive/sigma-v2-potong.zip',
                       '/content/drive/MyDrive/sigma-benih/sigma-v2-potong.zip'] if pathlib.Path(p).exists())
zipfile.ZipFile(ZIP).extractall('/content/potong')
DATA = '/content/potong/data.yaml'
cfg = yaml.safe_load(open(DATA)); cfg['path'] = '/content/potong'; yaml.safe_dump(cfg, open(DATA, 'w'))
NAMES = cfg['names']; print(NAMES)

In [ ]:
from pathlib import Path
from collections import Counter
for split in ['train', 'valid', 'test']:
    h = Counter()
    for f in Path('/content/potong', split, 'labels').glob('*.txt'):
        for b in f.read_text().splitlines():
            if b.strip(): h[(f.name.split('_')[0], NAMES[int(b.split()[0])])] += 1
    print(f'{split:6s}', dict(sorted(h.items())))

In [ ]:
from ultralytics import YOLO
IMGSZ = 1664
UMUM = dict(data=DATA, imgsz=IMGSZ, epochs=200, patience=80, batch=4, cache='ram',
            degrees=180, fliplr=0.5, flipud=0.5,      # benih menghadap ke mana saja
            hsv_h=0.0, hsv_s=0.2, hsv_v=0.2,          # warna = ciri cacat
            scale=0.05, translate=0.1,                # ukuran = ciri cacat, jarak kamera tetap
            mosaic=1.0, close_mosaic=30, copy_paste=0.5, mixup=0.0,
            cls=1.5,                                  # tekan kesalahan kelas (cacat minoritas)
            project=SIMPAN, exist_ok=True, seed=0, deterministic=True, plots=True)
hasil = {}
for nama, bobot in [('v2_y26s', 'yolo26s-seg.pt'), ('v2_y11s', 'yolo11s-seg.pt')]:
    try:
        YOLO(bobot).train(name=nama, **UMUM)
        m = YOLO(f'{SIMPAN}/{nama}/weights/best.pt').val(data=DATA, split='val', imgsz=IMGSZ, plots=False, verbose=False)
        hasil[nama] = m.seg.map50
        print(nama, 'valid mask mAP50 =', round(m.seg.map50, 3))
    except Exception as e:
        print(nama, 'GAGAL:', e)
PILIH = max(hasil, key=hasil.get); print('dipilih:', PILIH, hasil)

In [ ]:
# Pencocokan per benih: tiap benih label dicocokkan ke prediksi (IoU kotak > 0.5).
# Deteksi ganda (dua prediksi menutup benih yang sama, IoU > 0.5) dibuang, sisakan yang paling yakin.
import numpy as np, pandas as pd

def kotak_label(f, s):
    out = []
    for b in f.read_text().splitlines():
        if b.strip():
            v = np.array(b.split()[1:], float).reshape(-1, 2) * s
            out.append((int(b.split()[0]), [*v.min(0), *v.max(0)]))
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy; return i / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i + 1e-9)

def nilai(model, split, conf):
    baris = []
    for img in sorted(Path('/content/potong', split, 'images').glob('*.jpg')):
        gt = kotak_label(Path('/content/potong', split, 'labels', img.stem + '.txt'), IMGSZ)
        r = model.predict(str(img), imgsz=IMGSZ, conf=conf, verbose=False)[0]
        pred = sorted(zip(r.boxes.conf.tolist(), r.boxes.cls.int().tolist(), r.boxes.xyxy.tolist()), reverse=True)
        unik = []
        for p in pred:
            if all(iou(p[2], q[2]) <= 0.5 for q in unik): unik.append(p)
        pakai, cm = set(), Counter()
        for k, g in gt:
            j = max(((iou(g, p[2]), i) for i, p in enumerate(unik) if i not in pakai), default=(0, -1))
            if j[0] > 0.5: pakai.add(j[1]); cm[(NAMES[k], NAMES[unik[j[1]][1]])] += 1
            else: cm[(NAMES[k], 'terlewat')] += 1
        baris.append(dict(berkas=img.stem, varietas=img.stem.split('_')[0],
            baik_benar=sum(k == 0 for k, _ in gt), cacat_benar=sum(k == 1 for k, _ in gt),
            baik_pred=sum(p[1] == 0 for p in unik), cacat_pred=sum(p[1] == 1 for p in unik),
            cacat_tepat=cm[('cacat', 'cacat')], baik_tepat=cm[('baik', 'baik')],
            terlewat=cm[('baik', 'terlewat')] + cm[('cacat', 'terlewat')], berlebih=len(unik) - len(pakai)))
    return pd.DataFrame(baris)

terbaik = YOLO(f'{SIMPAN}/{PILIH}/weights/best.pt')
skor = {}
for c in [0.15, 0.25, 0.35, 0.5]:
    d = nilai(terbaik, 'valid', c)
    benar = d.baik_tepat.sum() + d.cacat_tepat.sum(); total = d.baik_benar.sum() + d.cacat_benar.sum()
    skor[c] = (benar - d.berlebih.sum()) / total
    print(f'conf={c}: akurasi per benih={benar/total:.3f}  cacat tepat={d.cacat_tepat.sum()}/{d.cacat_benar.sum()}  berlebih={d.berlebih.sum()}  terlewat={d.terlewat.sum()}')
CONF = max(skor, key=skor.get); print('conf dipilih dari valid:', CONF)

In [ ]:
# Laporan TEST (belum pernah dipakai memilih apa pun)
m = terbaik.val(data=DATA, split='test', imgsz=IMGSZ, project=SIMPAN, name=PILIH + '_test', exist_ok=True)
print(f'mask mAP50 {m.seg.map50:.3f} | mAP50-95 {m.seg.map:.3f}')
for i, c in enumerate(m.seg.ap_class_index):
    print(f'{m.names[int(c)]:6s} presisi={m.seg.p[i]:.3f} recall={m.seg.r[i]:.3f} mAP50={m.seg.ap50[i]:.3f}')
d = nilai(terbaik, 'test', CONF)
d['prop_cacat_benar'] = (d.cacat_benar / (d.baik_benar + d.cacat_benar)).round(3)
d['prop_cacat_pred'] = (d.cacat_pred / (d.baik_pred + d.cacat_pred).clip(lower=1)).round(3)
print(d.to_string(index=False)); print()
print(d.groupby('varietas')[['baik_benar','baik_pred','cacat_benar','cacat_pred','cacat_tepat','terlewat','berlebih']].sum())
benar = d.baik_tepat.sum() + d.cacat_tepat.sum(); total = d.baik_benar.sum() + d.cacat_benar.sum()
print(f'TEST conf={CONF}: akurasi per benih {benar/total:.3f}, cacat tepat {d.cacat_tepat.sum()}/{d.cacat_benar.sum()}, berlebih {d.berlebih.sum()}')

In [ ]:
from IPython.display import Image, display
contoh = sorted(Path('/content/potong/test/images').glob('VA_*'))[0]
terbaik.predict(str(contoh), imgsz=IMGSZ, conf=CONF, save=True, project=SIMPAN, name=PILIH + '_prediksi', exist_ok=True)
display(Image(filename=f'{SIMPAN}/{PILIH}_prediksi/{contoh.name}', width=800))

In [ ]:
berkas = terbaik.export(format='onnx', imgsz=IMGSZ)
print('Tersimpan:', berkas, '| conf:', CONF, '| input: potongan 1664 px skala asli di sekitar benih')